# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import testis_model as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from testis_model.metrics import init_time, last_time, min_time, max_time, freq_at, size, avg_freq_change, run_likelihood, n_transient

In [2]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
# clone_names = load_clone_names()

In [3]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['n500_r15_y6', 'r36_rp5_mimic', 'n500_r1_y6', 'r36_rp15', 'n1500_r36_y6', 'r36_rp1.5', 'r36_rp12', 'n500_r15_y9', 'r36_rp3.5_mimic', 'n1000_r1_y3', 'r18_rp3.5_mimic', 'r36_rp7_mimic', 'r36_rp13', 'n500_r1_y9', 'r36_rp14', 'n1500_r36_y9', 'r36_rp25', 'n1000_r15_y6', 'r36_rp2.5_mimic', 'r36_rp6', 'n1000_r0_y3', 'r36_rp3_mimic', 'r36_rp1', 'r36_rp8', 'r18_rp2.5_mimic', 'n500_r4_y6', 'n500_r0_y9', 'n1000_r4_y3', 'r36_rp9', 'n1000_r15_y9', 'r36_rp0', 'r36_rp7', 'n500_r0_y6', 'n500_r4_y9', 'r36_rp25_mimic', 'r18_rp8_mimic', 'r36_rp1_mimic', 'n1000_r36_y3', 'r18_rp0.5_mimic', 'n1500_r4_y9', 'r36_rp0.5_mimic', 'n1500_r0_y6', 'r36_rp2.5', 'r36_rp6_mimic', 'r36_rp4_mimic', 'r36_rp20_mimic', 'n1500_r0_y9', 'n1500_r4_y6', 'n500_r36_y3', 'r36_rp36', 'n1500_r1_y9', 'r36_rp0_mimic', 'n1500_r15_y3', 'r18_rp9_mimic', 'r36_rp30', 'r18_rp1.5_mimic', 'r36_rp2_mimic', 'n1500_r1_y6', 'r36_rp1.5_mimic', 'r36_rp3.5', 'r36_rp20', 'r36_rp11', 'r18_rp4.5_mimic', 'r18_rp13_mimic', 'r18_rp7_m

In [4]:
'r36_rp_5_mimic' 

'r36_rp_5_mimic'

In [5]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

for i, df in enumerate(long_dfs):
    if "duplicate" not in df.columns:
        df["duplicate"] = 0
        long_dfs[i] = df

In [6]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'duplicate', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 118812.


init_time  init_freq       r_opt  \
scenario    fish duplicate clone                                     
n500_r15_y6 0    0         0            6.0   0.427966   12.747156   
                           1            6.0   0.324859   10.220243   
                           2            6.0   0.177966    6.441394   
                           3            6.0   0.069209  883.424833   
                 1         0            6.0   0.368644    2.125072   

                                       l_max      lower      upper      l_null  
scenario    fish duplicate clone                                                
n500_r15_y6 0    0         0      -98.674588   5.224859  38.116989 -534.387357  
                           1     -104.051677   3.849996  36.337072 -231.840796  
                           2     -100.084527   2.439000  23.027977 -262.053325  
                           3      -16.283011  11.724790        NaN -225.089344  
                 1         0      -96.528094   0.393294  10.174459 -107.212870

In [7]:
# Specify order of columns
columns = ['scenario', 'fish', 'duplicate', 'clone', 'init_time', 'init_freq', 'last_time',
           'min_time', 'max_time', 'max_freq', 'n_transient', 'size', 'avg_freq_change', 'r_opt',
           'l_max', 'lower', 'upper', 'l_null']

rows = []
for (scenario, fish, duplicate, clone), data in tqdm(clone_data.items()):
    try:
        if (scenario, fish, duplicate, clone) in long_results.index:
            row = long_results.loc[scenario, fish, duplicate, clone].to_dict()
        else:
            row = {col: pd.NA for col in columns}
            it = init_time(data)
            if it is None: continue
            else: row['init_time'] = it

            row['init_freq'] = freq_at(data, row['init_time'])
    
        row['scenario'] = scenario
        row['fish'] = fish
        row['duplicate'] = duplicate
        row['clone'] = clone
        
        row['last_time'] = last_time(data)
        row['min_time'] = min_time(data)
        row['max_time'] = max_time(data)
        row['max_freq'] = freq_at(data, row['max_time'])
        row['n_transient'] = n_transient(data)
        row['size'] = size(data)
            
        row['avg_freq_change'] = avg_freq_change(data)
        
        rows.append(row)

    except Exception as e:
        print(f'Error for scenario={scenario}, fish={fish}, clone={clone}')
        raise

100%|██████████████████████████████████| 164547/164547 [03:56<00:00, 694.82it/s]


In [8]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [9]:
num_na = len(df[df['r_opt'].isna()])
num_fail = len(df[df['l_max'] < -1e5])

print(f'{num_na} rows have r_opt == NA, and {num_fail} have l_max < -1e5 (indicating a failure to converge).')

0 rows have r_opt == NA, and 0 have l_max < -1e5 (indicating a failure to converge).


In [10]:
df[df['scenario'] == 'full_pipeline_MP0.95']

,scenario,fish,duplicate,clone,init_time,init_freq,last_time,min_time,max_time,max_freq,n_transient,size,avg_freq_change,r_opt,l_max,lower,upper,l_null
75764,full_pipeline_MP0.95,1,0,0,4.86,0.772226,5.76,4.86,20.53,0.000000,2,0.139761,0.110213,301.663873,-30.246226,75.677664,1445.212250,-6814.042119
75765,full_pipeline_MP0.95,1,0,1,4.86,0.171932,20.53,4.86,20.53,0.803185,11,0.723035,-0.040284,29.683417,-156.410977,14.239943,72.309660,-2422.535841
75766,full_pipeline_MP0.95,1,0,2,4.86,0.020573,5.76,4.86,20.53,0.000000,2,0.004476,-0.005715,12.739770,-20.977551,2.028941,121.773361,-126.779602
75767,full_pipeline_MP0.95,1,0,3,4.86,0.016165,20.53,4.86,20.53,0.068259,11,0.046540,-0.003324,4.999702,-113.128944,2.129082,12.484378,-183.867316
75768,full_pipeline_MP0.95,1,0,4,4.86,0.011756,18.40,4.86,20.53,0.000000,10,0.025650,-0.000365,3.870731,-90.582544,1.504296,10.264998,-133.345818
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76032,full_pipeline_MP0.95,8,0,6,4.86,0.074219,12.10,4.86,12.10,0.043750,6,0.051917,0.004208,0.430857,-53.419156,0.070747,2.710758,-59.173517
76033,full_pipeline_MP0.95,8,0,7,4.86,0.039062,12.10,4.86,12.10,0.020313,6,0.031069,0.002590,0.359968,-54.493421,0.025173,2.877659,-57.512529
76034,full_pipeline_MP0.95,8,0,8,4.86,0.025000,12.10,4.86,12.10,0.021875,6,0.020847,0.000432,0.251190,-49.271662,0.000000,3.217959,-49.492534
76035,full_pipeline_MP0.95,8,0,9,4.86,0.014063,12.10,4.86,12.10,0.009375,6,0.008339,0.000647,0.517030,-42.473531,0.000000,2.995717,-44.321423
